In [4]:
!git clone https://github.com/endee-io/endee.git

Cloning into 'endee'...
remote: Enumerating objects: 2804, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 2804 (delta 18), reused 9 (delta 9), pack-reused 2781 (from 2)
Receiving objects: 100% (2804/2804), 3.63 MiB | 2.96 MiB/s, done.
Resolving deltas: 100% (1449/1449), done.


In [5]:
!pip install sentence-transformers numpy

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [7]:
class EndeeDB:
    def __init__(self):
        self.texts = []
        self.vectors = []

    def add(self, text, vector):
        self.texts.append(text)
        self.vectors.append(vector)

    def search(self, query_vector, top_k=2):
        similarities = []

        for vec in self.vectors:
            sim = np.dot(vec, query_vector) / (
                np.linalg.norm(vec) * np.linalg.norm(query_vector)
            )
            similarities.append(sim)

        top_indices = np.argsort(similarities)[-top_k:][::-1]

        return [(self.texts[i], similarities[i]) for i in top_indices]

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')
db = EndeeDB()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
documents = [
    "Artificial Intelligence is the future",
    "Machine Learning is a part of AI",
    "Deep Learning uses neural networks",
    "Python is popular for AI",
    "Vector databases are used in semantic search"
]

embeddings = model.encode(documents)

for doc, emb in zip(documents, embeddings):
    db.add(doc, emb)

print("Data stored successfully!")

Data stored successfully!


In [15]:
query = "What is AI?"

# FIX: always use [query] and take first element
query_embedding = model.encode([query])[0]

results = db.search(query_embedding, top_k=2)

print("Top Results:")
for text, score in results:
    print(f"{text} (score: {score:.4f})")

Top Results:
Machine Learning is a part of AI (score: 0.6561)
Python is popular for AI (score: 0.6035)


In [16]:
def generate_answer(query):
    query_embedding = model.encode([query])[0]  # FIXED

    results = db.search(query_embedding, top_k=2)

    context = " ".join([res[0] for res in results])

    return f"Answer based on data: {context}"

In [17]:
while True:
    q = input("Ask something (type 'exit' to stop): ")

    if q.lower() == "exit":
        print("Stopped.")
        break

    try:
        print(generate_answer(q))
    except Exception as e:
        print("Error:", e)

Ask something (type 'exit' to stop): exit
Stopped.


In [19]:
def search(self, query_vector, top_k=2):
    similarities = []

    for vec in self.vectors:
        sim = np.dot(vec, query_vector) / (
            np.linalg.norm(vec) * np.linalg.norm(query_vector)
        )
        similarities.append(sim)

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    return [(self.texts[i], similarities[i]) for i in top_indices]